In [ ]:
from google.colab import files

# upload json file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

Saving thread_level.json to thread_level.json


In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

# define message class
@dataclass
class Message:
    thread_id: Optional[str] = None
    message_id: str = ""
    subject: Optional[str] = None
    author: str = ""
    date_iso: Optional[str] = None
    body: str = ""
    url: str = ""
    in_reply_to: Optional[str] = None

# define thread class
@dataclass
class Thread:
    thread_id: str
    messages: List[Message]

In [ ]:
import json

# filters out unecessary fields in json files
def clean_message_fields(msg_dict):
    allowed_keys = {
        "thread_id", "message_id", "subject", "author",
        "date_iso", "body", "url", "in_reply_to"
    }
    return {k: v for k, v in msg_dict.items() if k in allowed_keys}


def load_query_and_messages(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    # TEMPORARY QUERY
    # based on the JSON file given by datascrape team
    query = "Summarize the following messages"
    messages = []

    # if 'messages' is found in thread, iterate over each instance of 'message'
    if isinstance(data, dict) and "messages" in data:
        for msg in data["messages"]:
            try:
                cleaned_msg = clean_message_fields(msg)
                messages.append(Message(**cleaned_msg))
            except TypeError as e:
                print(f"Error loading message: {e}")
    else:
        if isinstance(data, list):
            for thread in data:
                if "messages" in thread:
                    for msg in thread["messages"]:
                        try:
                            cleaned_msg = clean_message_fields(msg)
                            messages.append(Message(**cleaned_msg))
                        except TypeError as e:
                            print(f"Error loading message: {e}")
                else:
                    print("Warning: thread missing 'messages' key")
        else:
             print("Warning: Unexpected data format in JSON file.")


    return query, messages

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# RECOMMENDED EMBEDDING MODEL
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

class Retriever:
    # load documents to the retriever
    def add(self, documents: List[Message]):
        raise NotImplementedError

    # retrieve the most relevant documents
    def query(self, query_text: str, n_results: int = 5) -> List[Message]:
        raise NotImplementedError

# SEMANTIC EMBEDDING SEARCH FILTERING
class EmbeddingRetriever(Retriever):
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self._documents = []
        self._embeddings = []

    def add(self, documents: List[Message]):
        self._documents = documents
        texts = [doc.body for doc in documents]

        # make vectors unit-length to use cosine similarity
        self._embeddings = self.embedding_model.encode(texts, normalize_embeddings=True)

    def query(self, query_text: str, n_results: int = 5) -> List[Message]:
        # computes dot product similarity between the query and each stored message
        query_embedding = self.embedding_model.encode(query_text, normalize_embeddings=True)

        # sorts scores in descending order to get top score
        scores = np.dot(self._embeddings, query_embedding)
        top_indices = np.argsort(scores)[-n_results:][::-1]

        return [self._documents[i] for i in top_indices]

In [ ]:
# mock llm instruction
def build_prompt(query: str, retrieved_docs: list) -> str:
    # cleans long messages to make prompt shorter and readable
    def trim(text: str, max_len: int = 300):
      return text.replace("\n", " ").split(">", 1)[-1][:max_len].strip() + ("..." if len(text) > max_len else "")

    context = "\n".join(f"- {trim(doc.body)}" for doc in retrieved_docs)

    # TEMPORARY PROMPT
    # based on the JSON file given from datascrape team
    return (
    f"You are a scientific assistant helping researchers understand Amber simulation techniques.\n"
    f"Based on the email discussion below, explain how the HAMILTONIAN pointer works in replica exchange simulations.\n"
    f"Be clear, concise, and technically accurate.\n\n"
    f"Question: {query}\n\n"
    f"Email Messages:\n{context}\n\n"
    f"Answer:")

In [ ]:
from transformers import pipeline
from abc import ABC, abstractmethod

class BaseLLM(ABC):
    @abstractmethod
    def generate(self, prompt: str) -> str:
        pass

class HuggingFaceLLM(BaseLLM):
    def __init__(self, model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
        self.generator = pipeline("text-generation", model=model_name)

    def generate(self, prompt: str) -> str:
        result = self.generator(prompt, max_new_tokens=200, do_sample=False)
        return result[0]["generated_text"]

In [ ]:
def run_demo():
    query, messages = load_query_and_messages(filename)

    retriever = EmbeddingRetriever(embedding_model)
    retriever.add(messages)
    retrieved_docs = retriever.query(query, n_results=3)

    prompt = build_prompt(query, retrieved_docs)
    llm = HuggingFaceLLM()
    answer = llm.generate(prompt)

    print("Generated Answer:\n", answer)

run_demo()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated Answer:
 You are a scientific assistant helping researchers understand Amber simulation techniques.
Based on the email discussion below, explain how the HAMILTONIAN pointer works in replica exchange simulations.
Be clear, concise, and technically accurate.

Question: Summarize the following messages

Email Messages:
- On Tue, Feb 27, 2024, Adeleh Mokhles Gerami via AMBER wrote: [request to unsubscribe] Please visit http://lists.ambermd.org/mailman/listinfo/amber . ...dac
- Sent: Thursday, March 21, 2024 2:30 PM To: amber.ambermd.org Subject: [AMBER] GAMESS QM optimization protocols Hello Peers, MD experts, and fellow modelers, This is not a AMBER question, but just curious to know what you'll think about using GAMESS for a QM optimization and partial charge calculati...
- wrote: > Dear AMBER community, > > To get the total electrostatic energy from ESANDER output, do I need to > sum the [elec] and [elec14] columns? Or, is the [elec14] contribution > already included in the [e